# MLflow Autologging with PyTorch

**Notebook 2 of 3** · Demo for the "MLflow: Autologging & Zero-Config" talk

Here's the catch most tutorials skip: **`mlflow.autolog()` does not work on a raw PyTorch training loop.** A hand-written loop has no standard hook for MLflow to grab.

The fix is **PyTorch Lightning**, which gives autolog the structure it needs. This notebook shows both:
- Part A: the Lightning way (autolog works).
- Part B: the raw-loop way (manual logging).

Find more at [thamu.dev](https://thamu.dev/).

## Setup

In [ ]:
# uv:   !uv pip install "mlflow>=3.1" torch lightning torchvision
# pip:  !pip install "mlflow>=3.1" torch lightning torchvision

import mlflow, torch, lightning as L
print("mlflow    :", mlflow.__version__)
print("torch     :", torch.__version__)
print("lightning :", L.__version__)

## Part A — The Lightning way (autolog works)

We build a tiny classifier on synthetic data so it runs fast on CPU. The key line is `mlflow.autolog()`. It detects Lightning and logs metrics, params, and checkpoints during `trainer.fit()`.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

# Synthetic 2-class dataset (fast, CPU-friendly)
torch.manual_seed(42)
N, D = 1000, 20
X = torch.randn(N, D)
w = torch.randn(D, 1)
y = (X @ w + 0.1 * torch.randn(N, 1) > 0).long().squeeze()
train_loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)

In [ ]:
class LitClassifier(L.LightningModule):
    def __init__(self, in_dim=20, hidden=32, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()   # Lightning records these; autolog picks them up
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, 2)
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("train_loss", loss)   # surfaced by autolog
        self.log("train_acc", acc)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("pytorch-lightning-autolog-demo")
mlflow.autolog()   # detects Lightning; logs metrics, params, checkpoints

model = LitClassifier(in_dim=D, hidden=32, lr=1e-3)
trainer = L.Trainer(max_epochs=5, enable_progress_bar=True, logger=False)
trainer.fit(model, train_loader)   # run tracked automatically

## Part B — The raw PyTorch loop (manual logging)

If you don't use Lightning, autolog has nothing to hook into. You wrap the run yourself and call `mlflow.log_metric()` where it matters. More code, same destination.

In [ ]:
net = nn.Sequential(nn.Linear(D, 32), nn.ReLU(), nn.Linear(32, 2))
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

mlflow.autolog(disable=True)  # make the contrast explicit: autolog off here

with mlflow.start_run(run_name="raw-loop-manual"):
    mlflow.log_params({"hidden": 32, "lr": 1e-3, "epochs": 5, "framework": "raw-pytorch"})
    for epoch in range(5):
        running = 0.0
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(net(xb), yb)
            loss.backward()
            opt.step()
            running += loss.item()
        avg = running / len(train_loader)
        mlflow.log_metric("train_loss", avg, step=epoch)   # you log, by hand
        print(f"epoch {epoch}  loss {avg:.4f}")

## Open the MLflow UI

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db   # macOS: add --port 5001 if 5000 is taken by AirPlay
```

Open the experiment and compare:
- The **Lightning** run: metrics, params, and checkpoints appeared with no logging code.
- The **raw-loop** run: only the metrics you logged by hand.

**Takeaway:** autolog for PyTorch is really a Lightning feature. Reach for Lightning when you want zero-config; use manual `log_metric()` when you're on a custom loop.

## Recap

- `mlflow.autolog()` + PyTorch Lightning = zero-config tracking.
- Raw PyTorch loops need manual `mlflow.log_metric()` / `log_params()`.
- Same UI, same comparison, different amount of code.

**Next:** Notebook 3 crosses into GenAI, where the unit of work is a *trace*, not a `fit()`.